In [1]:
import requests

In [3]:
url = "https://www.hvakosterstrommen.no/api/v1/prices/2026/10-06_NO1.json"
response = requests.get(url)
print(response.status_code)

200


In [4]:
data = response.json()
print(len(data))
print(data[0])

24
{'NOK_per_kWh': 1.40463, 'EUR_per_kWh': 0.12968, 'EXR': 10.8315, 'time_start': '2026-10-06T00:00:00+02:00', 'time_end': '2026-10-06T01:00:00+02:00'}


In [5]:
df = pd.DataFrame(data)
df.head()

,NOK_per_kWh,EUR_per_kWh,EXR,time_start,time_end
0,1.40463,0.12968,10.8315,2026-10-06T00:00:00+02:00,2026-10-06T01:00:00+02:00
1,1.39813,0.12908,10.8315,2026-10-06T01:00:00+02:00,2026-10-06T02:00:00+02:00
2,1.38383,0.12776,10.8315,2026-10-06T02:00:00+02:00,2026-10-06T03:00:00+02:00
3,1.37820,0.12724,10.8315,2026-10-06T03:00:00+02:00,2026-10-06T04:00:00+02:00
4,1.38762,0.12811,10.8315,2026-10-06T04:00:00+02:00,2026-10-06T05:00:00+02:00


In [6]:
print(len(df) == 24)
print(df.isnull().sum().sum())
print(df.duplicated().sum())

True
0
0


In [7]:
def get_prices(date, area):
    url = f"https://www.hvakosterstrommen.no/api/v1/prices/{date[:4]}/{date[5:]}_{area}.json"
    response = requests.get(url)
    response.raise_for_status()
    df = pd.DataFrame(response.json())
    df["area"] = area
    return df

get_prices("2026-10-06", "NO2").head()

,NOK_per_kWh,EUR_per_kWh,EXR,time_start,time_end,area
0,1.49009,0.13757,10.8315,2026-10-06T00:00:00+02:00,2026-10-06T01:00:00+02:00,NO2
1,1.48933,0.13750,10.8315,2026-10-06T01:00:00+02:00,2026-10-06T02:00:00+02:00,NO2
2,1.47211,0.13591,10.8315,2026-10-06T02:00:00+02:00,2026-10-06T03:00:00+02:00,NO2
3,1.46561,0.13531,10.8315,2026-10-06T03:00:00+02:00,2026-10-06T04:00:00+02:00,NO2
4,1.48543,0.13714,10.8315,2026-10-06T04:00:00+02:00,2026-10-06T05:00:00+02:00,NO2


In [9]:
areas = ["NO1", "NO2", "NO3", "NO4", "NO5"]
frames = [get_prices("2026-10-06", area) for area in areas]
all_prices = pd.concat(frames, ignore_index=True)
print(len(all_prices))
all_prices.groupby("area")["NOK_per_kWh"].mean()

120


area
NO1    1.511382
NO2    1.619513
NO3    0.790027
NO4    0.099238
NO5    1.334613
Name: NOK_per_kWh, dtype: float64

In [11]:
def get_prices(date, area):
    url = f"https://www.hvakosterstrommen.no/api/v1/prices/{date[:4]}/{date[5:]}_{area}.json"
    response = requests.get(url)
    response.raise_for_status()
    df = pd.DataFrame(response.json())
    df["area"] = area
    return df

get_prices("2026-10-06", "NO1").head(24)

,NOK_per_kWh,EUR_per_kWh,EXR,time_start,time_end,area
0,1.40463,0.12968,10.8315,2026-10-06T00:00:00+02:00,2026-10-06T01:00:00+02:00,NO1
1,1.39813,0.12908,10.8315,2026-10-06T01:00:00+02:00,2026-10-06T02:00:00+02:00,NO1
2,1.38383,0.12776,10.8315,2026-10-06T02:00:00+02:00,2026-10-06T03:00:00+02:00,NO1
3,1.37820,0.12724,10.8315,2026-10-06T03:00:00+02:00,2026-10-06T04:00:00+02:00,NO1
4,1.38762,0.12811,10.8315,2026-10-06T04:00:00+02:00,2026-10-06T05:00:00+02:00,NO1
5,1.40322,0.12955,10.8315,2026-10-06T05:00:00+02:00,2026-10-06T06:00:00+02:00,NO1
6,1.45445,0.13428,10.8315,2026-10-06T06:00:00+02:00,2026-10-06T07:00:00+02:00,NO1
7,1.50450,0.13890,10.8315,2026-10-06T07:00:00+02:00,2026-10-06T08:00:00+02:00,NO1
8,1.53829,0.14202,10.8315,2026-10-06T08:00:00+02:00,2026-10-06T09:00:00+02:00,NO1
9,1.54392,0.14254,10.8315,2026-10-06T09:00:00+02:00,2026-10-06T10:00:00+02:00,NO1


In [12]:
date = "2026-10-06"
os.makedirs("data/raw", exist_ok=True)

for area in areas:
    url = f"https://www.hvakosterstrommen.no/api/v1/prices/{date[:4]}/{date[5:]}_{area}.json"
    response = requests.get(url)
    response.raise_for_status()
    with open(f"data/raw/prices_{date}_{area}.json", "w", encoding="utf-8") as f:
        f.write(response.text)

os.listdir("data/raw")

['prices_2026-10-06_NO1.json',
 'prices_2026-10-06_NO2.json',
 'prices_2026-10-06_NO3.json',
 'prices_2026-10-06_NO4.json',
 'prices_2026-10-06_NO5.json']

In [13]:
%%writefile fetch_prices.py
import os
import sys
from datetime import date

import requests

AREAS = ["NO1", "NO2", "NO3", "NO4", "NO5"]


def fetch_day(day):
    os.makedirs("data/raw", exist_ok=True)
    for area in AREAS:
        url = f"https://www.hvakosterstrommen.no/api/v1/prices/{day[:4]}/{day[5:]}_{area}.json"
        response = requests.get(url)
        response.raise_for_status()
        rows = response.json()
        if len(rows) not in (23, 24, 25):
            raise ValueError(f"{area} {day}: unexpected row count {len(rows)}")
        with open(f"data/raw/prices_{day}_{area}.json", "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"saved {area} {day}: {len(rows)} rows")


if __name__ == "__main__":
    day = sys.argv[1] if len(sys.argv) > 1 else date.today().isoformat()
    fetch_day(day)


Writing fetch_prices.py


In [14]:
!python fetch_prices.py 2026-10-05

saved NO1 2026-10-05: 24 rows
saved NO2 2026-10-05: 24 rows
saved NO3 2026-10-05: 24 rows
saved NO4 2026-10-05: 24 rows
saved NO5 2026-10-05: 24 rows
